In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

HOSP = Path("../data/mimiciv-3.1/hosp")
patients = pd.read_csv(HOSP / "patients.csv")
admissions = pd.read_csv(HOSP / "admissions.csv")
diagnoses_icd = pd.read_csv(HOSP / "diagnoses_icd.csv")
d_icd_diagnoses = pd.read_csv(HOSP / "d_icd_diagnoses.csv")

In [ ]:
display(patients.shape, patients.head(), patients.info(), patients.isna().sum(),
        patients.nunique(), patients.describe(), patients["gender"].value_counts())

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(patients["anchor_age"], bins=range(18, 93), edgecolor="white", color="steelblue")
ax.axvline(65, color="red", linestyle="--", label="65 years")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Number of patients")
ax.set_title("Anchor age distribution of MIMIC-IV patients")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
display(admissions.shape, admissions.head(), admissions.info(), admissions.isna().sum(),
        admissions.nunique(), admissions["admission_type"].value_counts(), admissions["hospital_expire_flag"].value_counts())


counts = admissions["admission_type"].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(counts.index, counts.values, color="steelblue")
ax.invert_yaxis()                       
ax.set_xlabel("Number of admissions")
ax.set_title("Admissions by admission type")
ax.grid(axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
display(diagnoses_icd.shape, diagnoses_icd.head(), diagnoses_icd.info(), diagnoses_icd.isna().sum(),
        diagnoses_icd.nunique(), diagnoses_icd["icd_version"].value_counts())

In [ ]:
display(d_icd_diagnoses.shape, d_icd_diagnoses.head(), d_icd_diagnoses.info(), d_icd_diagnoses.isna().sum(),
        d_icd_diagnoses.nunique(), d_icd_diagnoses["long_title"].value_counts())

In [ ]:
stays = pd.read_parquet("../data/processed/admissions_65plus.parquet")
print(len(stays))
diag_65 = diagnoses_icd[diagnoses_icd["hadm_id"].isin(stays["hadm_id"])].copy()

display(diag_65["icd_version"].value_counts())
display(diag_65.groupby("hadm_id").size().describe())
display(print(len(stays) - diag_65["hadm_id"].nunique()))

top = diag_65.groupby(["icd_code", "icd_version"]).size().reset_index(name="count")
top = top.sort_values("count", ascending=False).head(20)
#display(top_20_common)

top = pd.merge(top, d_icd_diagnoses, on=["icd_code", "icd_version"], how="left")
display(top)


top["label"] = top["long_title"].str.slice(0, 50) + " (ICD-" + top["icd_version"].astype(str) + ")"

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top["label"], top["count"], color="steelblue")
ax.invert_yaxis()                      
ax.set_xlabel("Number of stays")
ax.set_title("20 most common diagnoses, stays of patients aged 65+")
ax.grid(axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


per_stay = diag_65.groupby("hadm_id").size()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(per_stay, bins=range(1, per_stay.max() + 2), edgecolor="white", color="steelblue")
ax.axvline(per_stay.median(), color="red", linestyle="--", label=f"median = {per_stay.median():.0f}")
ax.set_xlabel("Number of diagnoses per stay")
ax.set_ylabel("Number of stays")
ax.set_title("Diagnoses per stay, patients aged 65+")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

diag_65.to_parquet("../data/processed/diagnoses_65plus.parquet", index=False)